# P-Task9: Library Management System

**Problem:** library ko books, members aur borrowing records manage karne hain. Books borrow aur return hoti
hain, members ki limit hoti hai, late return par fine lagta hai aur alag membership tiers ke alag rules hain.

**Concepts used:** single + multilevel + multiple inheritance, `super()`, MRO, method overriding,
`abc` / `@abstractmethod`, duck typing, dunder methods (`__str__`, `__repr__`, `__len__`, `__eq__`, `__lt__`,
`__getitem__`, `__iter__`, `__contains__`).

**Tiers (inheritance chain):**
```
Member (abstract)
  └── StudentMember   (3 books, 14 din, Rs 10/din)
        └── PremiumMember   (6 books, 21 din, fine aadha)
              └── VIPMember   (10 books, 30 din, 3 din grace, fine ki limit Rs 100)
```

## 1. Exceptions aur Book class

In [1]:
from abc import ABC, abstractmethod
from datetime import date, timedelta


class LibraryError(Exception):
    pass


class BookNotAvailableError(LibraryError):
    pass


class BorrowLimitError(LibraryError):
    pass


class Book:
    def __init__(self, isbn, title, author, copies=1):
        self.isbn = isbn
        self.title = title
        self.author = author
        self.copies = copies

    def __str__(self):
        return f"{self.title} by {self.author} [{self.copies} copies]"

    def __repr__(self):
        return f"Book({self.isbn!r}, {self.title!r})"

    def __eq__(self, other):
        return isinstance(other, Book) and self.isbn == other.isbn

    def __lt__(self, other):          # title ke hisaab se sort hone ke liye
        return self.title < other.title

## 2. Members (inheritance)
`Member` abstract hai, matlab uska object nahi ban sakta, har tier ko apni limits batani hongi.

In [2]:
class Member(ABC):
    def __init__(self, member_id, name):
        self.member_id = member_id
        self.name = name
        self.borrowed = []            # isbn ki list

    @property
    @abstractmethod
    def max_books(self): ...

    @property
    @abstractmethod
    def loan_days(self): ...

    fine_per_day = 10

    def fine(self, days_late):
        """Base fine logic. Bachay isay override karte hain."""
        return max(0, days_late) * self.fine_per_day

    def __len__(self):
        return len(self.borrowed)

    def __lt__(self, other):
        return len(self) < len(other)

    def __str__(self):
        tier = type(self).__name__.replace("Member", "")
        return f"{self.member_id} {self.name:<8} [{tier}] {len(self)}/{self.max_books} books"

    def __repr__(self):
        return f"{type(self).__name__}({self.member_id!r}, {self.name!r})"


class StudentMember(Member):
    max_books = 3
    loan_days = 14


class PremiumMember(StudentMember):           # multilevel: Member -> Student -> Premium
    max_books = 6
    loan_days = 21

    def fine(self, days_late):                # overriding: parent ka fine aadha
        return super().fine(days_late) / 2


class Notifier:                                # mixin, multiple inheritance dikhane ke liye
    def notify(self, message):
        return f"[SMS to {self.name}] {message}"


class VIPMember(PremiumMember, Notifier):     # multiple inheritance
    max_books = 10
    loan_days = 30

    def fine(self, days_late):
        days_late -= 3                         # 3 din grace
        return min(super().fine(days_late), 100)   # aur fine ki limit Rs 100

### MRO (method resolution order)
Python kis class mein pehle method dhoondta hai, uska order:

In [3]:
[cls.__name__ for cls in VIPMember.__mro__]

['VIPMember', 'PremiumMember', 'StudentMember', 'Member', 'ABC', 'Notifier', 'object']

### Abstract class ka object nahi ban sakta

In [4]:
try:
    Member("X1", "Test")
except TypeError as e:
    print("TypeError:", e)

TypeError: Can't instantiate abstract class Member without an implementation for abstract methods 'loan_days', 'max_books'


### Fine ka tier wise farq (10 din late)

In [5]:
members_demo = [StudentMember("S1", "Ayesha"), PremiumMember("P1", "Bilal"), VIPMember("V1", "Fatima")]
for m in members_demo:                         # duck typing: sab ke paas .fine() hai
    print(f"{type(m).__name__:<15} 10 din late -> Rs {m.fine(10)}")
print("30 din late (VIP):", members_demo[2].fine(30), "(limit Rs 100 lag gayi)")

StudentMember   10 din late -> Rs 100
PremiumMember   10 din late -> Rs 50.0
VIPMember       10 din late -> Rs 35.0
30 din late (VIP): 100 (limit Rs 100 lag gayi)


## 3. Library class

In [6]:
class BorrowRecord:
    def __init__(self, isbn, member_id, borrowed_on, due_on):
        self.isbn = isbn
        self.member_id = member_id
        self.borrowed_on = borrowed_on
        self.due_on = due_on
        self.returned_on = None
        self.fine = 0

    def __str__(self):
        status = f"returned {self.returned_on}, fine Rs {self.fine:g}" if self.returned_on else "abhi issued"
        return f"{self.member_id} -> {self.isbn} (due {self.due_on}) {status}"


class Library:
    def __init__(self, name):
        self.name = name
        self.books = {}
        self.members = {}
        self.records = []

    def add_book(self, book):
        self.books[book.isbn] = book

    def add_member(self, member):
        self.members[member.member_id] = member

    # ---- dunder methods ----
    def __len__(self):
        return len(self.books)

    def __getitem__(self, isbn):               # library["B1"]
        return self.books[isbn]

    def __iter__(self):                        # for book in library
        return iter(sorted(self.books.values()))

    def __contains__(self, isbn):              # "B1" in library
        return isbn in self.books

    # ---- borrowing ----
    def borrow(self, member_id, isbn, on):
        member, book = self.members[member_id], self.books[isbn]
        if len(member) >= member.max_books:
            raise BorrowLimitError(f"{member.name} ki limit ({member.max_books}) poori ho chuki hai")
        if book.copies < 1:
            raise BookNotAvailableError(f"'{book.title}' abhi available nahi")
        book.copies -= 1
        member.borrowed.append(isbn)
        record = BorrowRecord(isbn, member_id, on, on + timedelta(days=member.loan_days))
        self.records.append(record)
        return record

    def return_book(self, member_id, isbn, on):
        member = self.members[member_id]
        record = next(r for r in self.records
                      if r.member_id == member_id and r.isbn == isbn and r.returned_on is None)
        record.returned_on = on
        record.fine = member.fine((on - record.due_on).days)
        member.borrowed.remove(isbn)
        self.books[isbn].copies += 1
        return record.fine

## 4. Library chalana

In [7]:
lib = Library("Smart District Library")
lib.add_book(Book("B1", "Python Basics", "Mark Lutz", 2))
lib.add_book(Book("B2", "Data Structures", "Sedgewick", 1))
lib.add_book(Book("B3", "Machine Learning", "Tom Mitchell", 1))
lib.add_book(Book("B4", "Algorithms", "CLRS", 3))
lib.add_book(Book("B5", "Operating Systems", "Silberschatz", 1))

lib.add_member(StudentMember("M1", "Ayesha"))
lib.add_member(PremiumMember("M2", "Bilal"))
lib.add_member(VIPMember("M3", "Fatima"))

print("Total books:", len(lib))
print("B2 library mein hai?", "B2" in lib)
print("B9 library mein hai?", "B9" in lib)
print(lib["B1"])
print("\nTitle ke hisaab se sorted:")
for book in lib:
    print("  ", book)

Total books: 5
B2 library mein hai? True
B9 library mein hai? False
Python Basics by Mark Lutz [2 copies]

Title ke hisaab se sorted:
   Algorithms by CLRS [3 copies]
   Data Structures by Sedgewick [1 copies]
   Machine Learning by Tom Mitchell [1 copies]
   Operating Systems by Silberschatz [1 copies]
   Python Basics by Mark Lutz [2 copies]


## 5. Borrow / return scenarios

In [8]:
start = date(2026, 10, 1)

for isbn in ["B1", "B2", "B3"]:
    lib.borrow("M1", isbn, start)
print(lib.members["M1"])

# 4th book par limit
try:
    lib.borrow("M1", "B4", start)
except BorrowLimitError as e:
    print("BorrowLimitError:", e)

# B2 ek hi copy thi aur Ayesha ke paas hai
try:
    lib.borrow("M2", "B2", start)
except BookNotAvailableError as e:
    print("BookNotAvailableError:", e)

lib.borrow("M2", "B1", start)
lib.borrow("M3", "B4", start)
print("Bilal aur Fatima ki books issue ho gayin")

M1 Ayesha   [Student] 3/3 books
BorrowLimitError: Ayesha ki limit (3) poori ho chuki hai
BookNotAvailableError: 'Data Structures' abhi available nahi
Bilal aur Fatima ki books issue ho gayin


## 6. Returns aur fines
Return dates: Ayesha 28 din baad (14 din late), Bilal 31 din baad (10 din late),
Fatima 80 din baad (50 din late).

In [9]:
f1 = lib.return_book("M1", "B1", start + timedelta(days=28))
f2 = lib.return_book("M2", "B1", start + timedelta(days=31))
f3 = lib.return_book("M3", "B4", start + timedelta(days=80))
f4 = lib.return_book("M1", "B2", start + timedelta(days=10))      # waqt par

print("Ayesha (Student)  fine: Rs", f1)
print("Bilal  (Premium)  fine: Rs", f2)
print("Fatima (VIP)      fine: Rs", f3)
print("Ayesha waqt par   fine: Rs", f4)
print()
print(lib.members["M3"].notify("aap ki kitab wapas mil gayi, shukriya"))

Ayesha (Student)  fine: Rs 140
Bilal  (Premium)  fine: Rs 50.0
Fatima (VIP)      fine: Rs 100
Ayesha waqt par   fine: Rs 0

[SMS to Fatima] aap ki kitab wapas mil gayi, shukriya


## 7. Saare records aur members

In [10]:
for record in lib.records:
    print(record)

print()
for member in sorted(lib.members.values(), reverse=True):     # __lt__ se sorting
    print(member, "|", repr(member))

M1 -> B1 (due 2026-10-15) returned 2026-10-29, fine Rs 140
M1 -> B2 (due 2026-10-15) returned 2026-10-11, fine Rs 0
M1 -> B3 (due 2026-10-15) abhi issued
M2 -> B1 (due 2026-10-22) returned 2026-11-01, fine Rs 50
M3 -> B4 (due 2026-10-31) returned 2026-12-20, fine Rs 100

M1 Ayesha   [Student] 1/3 books | StudentMember('M1', 'Ayesha')
M2 Bilal    [Premium] 0/6 books | PremiumMember('M2', 'Bilal')
M3 Fatima   [VIP] 0/10 books | VIPMember('M3', 'Fatima')


## Conclusion
Teeno tiers ek hi chain mein bane (Student -> Premium -> VIP) aur har tier ne `fine()` apne tareeqe se override
kiya. Wohi 10 din late par Student ka fine Rs 100, Premium ka Rs 50 aur VIP ka Rs 35 aaya. Dunder methods se
library ko `len()`, `in`, `[]` aur `for` ke saath use kar sakte hain.